# Xenium mouse brain: curve test with slide effects

Wildtype vs. TgCRND8, all sections. For every gene, does its expression change differently with the distance to the dentate gyrus in the two conditions, measured against the variation between sections?

1. Quick pre-processing
2. The anchor and the distance to it
3. The fit: curve test on single cells, full-rank ADVI with empirical-Bayes slide effects
4. Results: diagnostics, ranked genes, volcano, p-value histogram, curves of the top genes
5. Clusters of the fitted curves

With several sections per condition, the slide effects (a random level and a random curve per section) separate what the conditions do from what happens to differ between sections. They need at least 3 sections in total.

In [ ]:
# One BLAS thread per process: the fits run in worker processes, and oversubscribed BLAS threads slow them down a lot.
# This has to happen before numpy is imported, so restart the kernel if numpy is already loaded.
import os

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import diffsvg as dvg

## Settings

**To fill in once the prepared file exists:** `DATA_PATH` and `SAMPLE`. Check that `CONDITION`, `ANCHOR_KEY` and `ANCHOR_GROUP` still match its columns.

* `SAMPLE`: the `obs` column naming the section of every cell. Each section must belong to one condition.
* `LAYER`: log1p of the library-normalised counts, unscaled. `tau` is defined on this scale; `sc.pp.scale` would give it a different meaning for every gene.
* `TAU`: the smallest shape difference that counts for the test, in slope-equivalent units. 0.1 means the fold change between the conditions drifts by about 10% across the distance range.
* `GENES`: `None` fits all genes. The p-value is per gene, so results on a subset are valid too, for example `GENES = list(adata.var_names[:12])` for a trial.
* `USE_BINNED`: `False` fits every cell. `True` fits per-slide contiguous bins weighted by their cell counts, which have the same likelihood as the cells. On legnini they gave the same statistic at about a twelfth of the cost (see `Full tutorial.ipynb`), so use them if the fit on cells takes too long.

In [ ]:
# On the cluster, xenium_advi_slides.sbatch sets the paths and the workers.
DATA_PATH = os.environ.get("DIFFSVG_DATA", "/home/lehnerl/Arbeit/data/HiWi/TODO_all_sections.h5ad")  # TODO: the prepared file
RESULTS_PATH = os.environ.get("DIFFSVG_RESULTS", "Xenium_ADVI_results.pkl")
SAMPLE = "sample"  # TODO: the obs column naming the section
CONDITION = "condition"
ANCHOR_KEY = "Dentate_Gyrus"
ANCHOR_GROUP = "anchor"
LAYER = "log1p_norm"
TAU = 0.1
N_WORKERS = int(os.environ.get("DIFFSVG_WORKERS", 4))
GENES = None
USE_BINNED = False
N_CLUSTERS = 5

## 1. Pre-processing

Cells with fewer than 10 transcripts are dropped before anything else, so they cannot set a section's farthest distance. `.X` keeps the counts.

In [ ]:
adata = sc.read_h5ad(DATA_PATH)
n_before = adata.obs.groupby([CONDITION, SAMPLE], observed=True).size()
sc.pp.filter_cells(adata, min_counts=10)

adata.layers[LAYER] = adata.X.copy()
sc.pp.normalize_total(adata, layer=LAYER)
sc.pp.log1p(adata, layer=LAYER)

conditions_per_slide = adata.obs.groupby(SAMPLE, observed=True)[CONDITION].nunique()
if (conditions_per_slide > 1).any():
    raise ValueError(f"Sections in more than one condition: {list(conditions_per_slide.index[conditions_per_slide > 1])}")
pd.DataFrame({"cells before": n_before, "cells after": adata.obs.groupby([CONDITION, SAMPLE], observed=True).size()})

## 2. Anchor and distance

`var_by_distance` gives every cell its distance to the closest dentate gyrus cell of its own section, then scales each section to 0-1 by its own range. All sections therefore cover the whole x-axis, without blank space at the far end. The price is that the same physical distance gets a slightly different x in different sections; the last column shows how far the sections reach. A section without anchor cells cannot be placed on this axis and drops out of the fit.

In [ ]:
sq.tl.var_by_distance(adata, cluster_key=ANCHOR_KEY, groups=ANCHOR_GROUP, library_key=SAMPLE, covariates=CONDITION)
design = adata.obsm["design_matrix"]

design.groupby([CONDITION, SAMPLE], observed=True).agg(
    fitted_cells=(ANCHOR_GROUP, "count"),
    anchor_cells=(ANCHOR_GROUP, lambda x: x.isna().sum()),
    farthest_cell=(f"{ANCHOR_GROUP}_raw", "max"),
)

In [ ]:
slides = list(design.groupby([CONDITION, SAMPLE], observed=True).size().index)
is_anchor = (adata.obs[ANCHOR_KEY] == ANCHOR_GROUP).to_numpy()
ncols = min(4, len(slides))
nrows = (len(slides) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.2 * nrows), squeeze=False, constrained_layout=True)
for ax in axes.ravel()[len(slides) :]:
    ax.set_visible(False)
for ax, (condition, slide) in zip(axes.ravel(), slides, strict=False):
    in_slide = (adata.obs[SAMPLE] == slide).to_numpy()
    xy = adata.obsm["spatial"][in_slide]
    scatter = ax.scatter(
        *xy[~is_anchor[in_slide]].T,
        c=design[ANCHOR_GROUP].to_numpy()[in_slide][~is_anchor[in_slide]],
        cmap="viridis", vmin=0, vmax=1, s=0.5, linewidth=0, rasterized=True,
    )  # fmt: skip
    ax.scatter(*xy[is_anchor[in_slide]].T, color="red", s=0.5, linewidth=0, rasterized=True)
    ax.set_title(f"{slide} ({condition})", fontsize=9)
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.colorbar(scatter, ax=axes, shrink=0.6, label="Distance to the dentate gyrus (scaled per section); anchor in red");

## 3. Fit

* **Model:** one spline per gene and condition, plus a random level and a random curve per section (`sample=SAMPLE`). The curves shown later are the population-level ones.
* **Inference:** the two scales of the slide effects are set per gene by empirical Bayes (the mode of their marginal posterior). ADVI then fits the rest with those scales fixed, started at the Laplace approximation and polished with Adam.
* **Test:** `test="parallel"`, whether the two curves differ by more than a shift in level. The p-value uses an F reference with `n_slides - 2` denominator degrees of freedom.

**Runtime:** not measured on this data. On legnini (41,000 cells, 16 slides) a fit with slide effects took about 90 s per gene on one worker, about five times the fit without them. On the two Xenium sections (113,000 cells) the fit without slide effects took 40 s per gene on one worker. So expect several minutes per gene on one worker here, and hours for the whole panel with 12 workers. For a trial, shorten `GENES` in the settings. If it is too slow, set `USE_BINNED = True`.

In [ ]:
dvg.pp.bin_obs(
    adata, anchor=ANCHOR_GROUP, condition=CONDITION, genes=GENES, layer=LAYER, n_bins=50, window_radius=None, sample=SAMPLE
)
setup = dvg.tl.setup_model(adata)

t0 = time.perf_counter()
dvg.tl.fit_model(
    adata, setup, n_workers=N_WORKERS, use_binned=USE_BINNED, inference="advi", slide_scales="empirical_bayes",
    test="parallel", tau=TAU,
)  # fmt: skip
elapsed = time.perf_counter() - t0
print(f"{elapsed / 60:.1f} min, {elapsed * N_WORKERS / len(setup.binned.genes):.0f} s per gene on one worker")

The fit takes long, so its tables are saved right away. Pre-processing plus this file is enough to continue later: put each table back with `adata.varm[key] = table` (the genes follow `adata.var_names`).

In [ ]:
RESULT_KEYS = [key for key in adata.varm if key.startswith("dsvg_")]
pd.to_pickle({key: pd.DataFrame(adata.varm[key]) for key in RESULT_KEYS}, RESULTS_PATH)
RESULT_KEYS

## 4. Results

### Did the approximation work?

`vi_khat` and `vi_ess` compare each gene's approximation with its true posterior. With slide effects the covariance the test is built on can be trusted for `vi_khat < 0.7` and `vi_ess > 0.1`; the ESS share falls with the number of parameters even for a good fit. Do not read the p-values of genes that fail.

In [ ]:
stats = adata.varm["dsvg_stats"].dropna(how="all").copy()
stats["vi_trusted"] = (stats["vi_khat"] < 0.7) & (stats["vi_ess"] > 0.1)

print(f"{stats['vi_trusted'].sum()} of {len(stats)} genes pass the variational diagnostics")
print(f"p < 0.05: {(stats['parallel_pval'] < 0.05).sum()}, BH-adjusted < 0.05: {(stats['parallel_pval_adj'] < 0.05).sum()}")
stats.loc[~stats["vi_trusted"], ["vi_khat", "vi_ess", "sample_level_sd", "sample_curve_sd", "parallel_pval"]]

### Ranked genes

Ordered by p-value, with ties (p-values that underflow to 0, or sit at 1) broken by the larger statistic. `sample_level_sd` and `sample_curve_sd` are the estimated differences between sections, in the level and in the curve.

In [ ]:
ranked = stats.sort_values(["parallel_pval", "parallel_stat"], ascending=[True, False])
columns = [
    "shape_diff", "slope_diff", "slope_diff_ci_low", "slope_diff_ci_high", "parallel_stat", "parallel_df_denominator",
    "parallel_pval", "parallel_pval_adj", "integral", "sample_level_sd", "sample_curve_sd", "vi_khat", "vi_ess", "vi_trusted",
]  # fmt: skip
ranked[columns].head(25)

### Volcano and p-value histogram

The curves are compared as condition 1 minus condition 0, with the condition labels in sorted order, so **`TgCRND8` is condition 0 and `wildtype` is condition 1**. A positive `slope_diff` means wildtype minus TgCRND8 rises with the distance; the cell prints the coding of this run.

With `tau > 0`, genes whose shape difference is clearly below `tau` get p-values near 1, so the histogram has a second peak there by design.

In [ ]:
print(adata.uns["dsvg_curves_condition_categories"])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
dvg.pl.plot_pvalue_histogram(stats, ax=axes[0])
dvg.pl.plot_volcano(stats, ax=axes[1], n_label=15)
fig.suptitle(f"Curve test with empirical-Bayes slide effects (tau = {TAU})")
fig.tight_layout();

### Curves of the top genes

The top genes that pass the diagnostics, by condition. The bands are quantiles of the posterior-predictive draws, which include the noise of a single cell, so on a fit to cells they are much wider than the uncertainty of the curve itself. Judge the shape difference by the test and the mean curves, not by whether the bands overlap.

In [ ]:
TOP_GENES = list(ranked.index[ranked["vi_trusted"]][:8])
dvg.pl.plot_prediction_curve(adata, gene=TOP_GENES);

## 5. Clusters of the fitted curves

`cluster_curves` runs k-means on the posterior means of the spline coefficients, in two ways:

* **Combined** (`by="average"`): the mean of the two condition curves, so genes group by their shared pattern along the distance.
* **Interaction** (`by="interaction"`): the difference between the conditions, so genes group by how the conditions differ. Genes without a difference collect in a flat cluster.

The interaction clusters share their source with the test (`beta_1`), so finding the flagged genes in the patterned clusters describes them and does not confirm them. With a panel this size the gene names do not fit in the plots; the table lists them.

In [ ]:
def cluster_table(key):
    """One row per cluster: its size, how many of its genes the test flags, and the genes."""
    labels = adata.var[key].dropna()
    return pd.DataFrame(
        {
            cluster: {
                "n": len(genes),
                "BH < 0.05": int((stats.loc[genes, "parallel_pval_adj"] < 0.05).sum()),
                "median shape_diff": round(float(stats.loc[genes, "shape_diff"].median()), 2),
                "genes": ", ".join(genes),
            }
            for cluster, genes in ((c, list(labels.index[labels == c])) for c in labels.cat.categories)
        }
    ).T


dvg.tl.cluster_curves(adata, by="average", n_clusters=N_CLUSTERS)
display(cluster_table("dsvg_cluster_average"))
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_average", show_legend=False);

In [ ]:
dvg.tl.cluster_curves(adata, by="interaction", n_clusters=N_CLUSTERS)
display(cluster_table("dsvg_cluster_interaction"))
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_interaction", show_legend=False);

The same interaction clusters, split by condition, with one row per cluster and the first condition on the left. Each gene is centred over both conditions together, so the gap between them stays visible.

In [ ]:
dvg.pl.plot_cluster_curves(adata, "dsvg_cluster_interaction", curves="conditions", show_legend=False);